# Groq API Playground and Error-Handling 

This notebook demonstrates how to:

- load a Groq API key from a local `.env` file
- create a chat completion request with the OpenAI-compatible Groq endpoint
- inspect API responses and errors
- handle common failures such as rate limits and authentication issues
- wrap repeated requests in a small helper function for safer usage

## Setup

Before running the notebook, create a `.env` file in the same folder and add:

```bash
GROQ_API_KEY="your_api_key_here"
```

The first code cell loads the environment variable and validates that the API key is present. The later cells test basic requests and handle the most common error cases returned by the Groq API.

## Notes

- This notebook uses the OpenAI Python client with Groq's OpenAI-compatible API endpoint.
- The examples are intentionally simple and meant for learning and debugging.
- Replace the model name if you want to try different Groq models available in your account.


In [45]:
from dotenv import load_dotenv
from openai import OpenAI
import os
import time


In [ ]:
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [33]:
client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [34]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain hypertension in simple terms for a patient."
        }
    ]
)

print(response.choices[0].message.content)

### What Is Hypertension?  
**Hypertension = “High Blood Pressure.”**  
It’s the medical name for when the force of blood pushing against the walls of your arteries is higher than it should be.

---

#### Why Does It Matter?  
- The arteries are like tubes that carry blood.  
- If the pressure inside those tubes is too high for long, they can wear out, causing problems in the heart, brain, kidneys, and other organs.  
- Often you don’t feel anything—hence the nickname **“the silent killer.”**

---

#### How Do Doctors Measure It?  
A blood‑pressure reading looks like two numbers, e.g. **120 / 80 mm Hg**.  

| Term | What It Means | Typical Ranges |
|------|---------------|----------------|
| **Systolic** (top number) | Pressure when your heart beats | <120 |
| **Diastolic** (bottom number) | Pressure when the heart is resting between beats | <80 |
| **Hypertension** | Systolic ≥ 130 **or** Diastolic ≥ 80 | Stage 1: 130–139 / 80–89  <br>Stage 2: ≥ 140 / ≥ 90 |

*If you’re reading 120/80

In [35]:
print(type(response))

<class 'openai.types.chat.chat_completion.ChatCompletion'>


In [36]:
print(response)

ChatCompletion(id='chatcmpl-5b2042c8-accf-4238-8af6-7fdffc28f51f', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='### What Is Hypertension?  \n**Hypertension = “High Blood Pressure.”**  \nIt’s the medical name for when the force of blood pushing against the walls of your arteries is higher than it should be.\n\n---\n\n#### Why Does It Matter?  \n- The arteries are like tubes that carry blood.  \n- If the pressure inside those tubes is too high for long, they can wear out, causing problems in the heart, brain, kidneys, and other organs.  \n- Often you don’t feel anything—hence the nickname **“the silent killer.”**\n\n---\n\n#### How Do Doctors Measure It?  \nA blood‑pressure reading looks like two numbers, e.g. **120\u202f/\u202f80\u202fmm\u202fHg**.  \n\n| Term | What It Means | Typical Ranges |\n|------|---------------|----------------|\n| **Systolic** (top number) | Pressure when your heart beats | <120 |\n| **Diastolic** (bottom 

## 1) Basic API request

This section validates that the environment is configured correctly and that the Groq endpoint responds to a simple chat request. It is the baseline check before handling more advanced failures.

In [37]:
try:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": "What are three common symptoms of diabetes?"
            }
        ]
    )

    print(response.choices[0].message.content)

except Exception as e:
    print("API request failed:", e)

API request failed: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}


In [38]:
models = client.models.list()

for model in models.data:
    print(model.id)

canopylabs/orpheus-v1-english
openai/gpt-oss-20b
qwen/qwen3.8-27b
whisper-large-v3
allam-2-7b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m


## 2) Rate limit handling

This example demonstrates how to catch a `RateLimitError` and pause before retrying. In production, this pattern helps when a model endpoint is busy or your API quota has been temporarily exhausted.

In [21]:
from openai import RateLimitError

print(RateLimitError)

<class 'openai.RateLimitError'>


In [40]:
try:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": "What are three common symptoms of diabetes?"
            }
        ]
    )

    print(response.choices[0].message.content)

except RateLimitError:
    print("Rate limit reached. Please wait and try again.")
    
except Exception as e:
    print("API request failed:", e)

Three common signs that people with diabetes may experience are:

1. **Excessive thirst (polydipsia)** – feeling constantly thirsty even after drinking fluids.  
2. **Frequent urination (polyuria)** – needing to pee many times per day, especially at night.  
3. **Unexplained weight loss** – losing weight even when you’re eating normally or you’re eating more than usual.

These symptoms arise because high blood sugar levels cause the kidneys to excrete excess glucose, pulling fluid from the body and stimulating thirst, and because the body can’t use glucose properly, it starts breaking down fat and muscle for energy. If you notice any of these symptoms, it’s a good idea to talk to a healthcare professional for testing and guidance.


In [49]:
def ask_llm(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            return response.choices[0].message.content

        except RateLimitError:
            print(f"Rate limit reached. Attempt {attempt + 1}/{max_retries}")
            
            if attempt < max_retries - 1:
                time.sleep(5)
            else:
                print("Maximum retries reached.")
                return None

        except Exception as e:
            print("API request failed:", e)
            return None

In [57]:
result = ask_llm("What are two common risk factors for hypertension?")

print(result)

Two common risk factors for developing high blood pressure (hypertension) are:

1. **High sodium (salt) intake** – Consuming too much salt can increase blood volume and raise blood pressure.
2. **Overweight or obesity** – Excess body weight puts additional strain on the cardiovascular system, often leading to elevated blood pressure.


## 3) Authentication handling

This section shows how to handle `AuthenticationError`, which usually means the API key is missing, invalid, or not authorized for the requested model. It is important to validate credentials before attempting more complex requests.

In [58]:
from openai import AuthenticationError

print(AuthenticationError)

<class 'openai.AuthenticationError'>


In [59]:
def ask_llm(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            return response.choices[0].message.content

        except RateLimitError:
            print(f"Rate limit reached. Attempt {attempt + 1}/{max_retries}")

            if attempt < max_retries - 1:
                time.sleep(5)
            else:
                print("Maximum retries reached.")
                return None

        except AuthenticationError:
            print("Authentication failed. Check your API key.")
            return None

        except Exception as e:
            print("API request failed:", e)
            return None

In [60]:
result = ask_llm("What are two common risk factors for hypertension? Explain in one sentence ")

print(result)

Two common risk factors for hypertension are excessive salt consumption and obesity, because high sodium intake raises blood volume and pressure while excess body weight increases cardiac output and arterial resistance.


In [61]:
result = ask_llm(
    "Explain why regular blood pressure monitoring is important for a patient with hypertension."
)

print(result)

**Why regular blood‑pressure (BP) monitoring matters for people with hypertension**

| What regular monitoring gives you | Why it’s important for your health |
|----------------------------------|------------------------------------|
| **An accurate picture of your BP over time** | Blood pressure fluctuates day‑to‑day. A single reading in a doctor’s office can be “white‑coat” hypertension (higher because of anxiety) or “masked” hypertension (normal in the office but high elsewhere). Repeated readings—especially at home—capture the true pattern and help your clinician decide what to do. |
| **Early detection of “out‑of‑range” values** | If BP climbs or dips far outside the target range, you’ll know sooner and can adjust medication or lifestyle before complications develop. |
| **Guidance on medication adjustments** | Your doctor uses trends, not a single snapshot, to tweak doses, add new drugs, or change a medication’s timing. |
| **Feedback for lifestyle changes** | BP often falls with